# Project Test Console — Colab

Reusable test notebook with settings in Google Colab's native form controls. No custom web UI or ipywidgets.

## Quick start
1. Edit the form fields in the runner cell (sections 1–6).
2. Run the cell. For private repos, add a Colab Secret (default GH_TOKEN) and enable access.
3. Use the HTML report's copyable text area, or download TXT/HTML and send the report to ChatGPT.

## Settings guide
- Project: profile, repository URL and branch/ref.
- Git access: Secret name, username, shallow/full clone and clone timeout.
- Runtime: test timeout, optional Clang/libc++ installation, continue on failures and keep workspace.
- Setup: up to three shell setup commands.
- Test matrix: up to five tests, each Name :: command :: needs=tool.
- Reuse: optionally mount Drive to load/save a JSON settings file. Keep this off if you prefer not to mount Drive.

## Tessera profile notes
Clang mock tests use libc++ to provide C++23 std::expected support. The notebook can install Clang and libc++ if enabled. GCC strict mock and GCC ASan/UBSan are separate tests.

Safety: setup and test commands execute in the Colab shell. Only run repositories and commands you trust. GitHub token is used only for clone and removed from the test environment. The report does not include token values.

## Output\nThe runner displays an HTML summary, failure details, an expandable issue list, and a full-report text area. Download the TXT or HTML report using the links at the bottom of the output.\n\n## Scope
Portable Linux mock tests do not validate real Windows/MSVC inline hooks, Detours, x64 unwind registration, DLL lifecycle, loader behavior or Windows-native tests. Those require Windows validation.


In [ ]:
#@title 🧪 Project Test Console — Configuration { display-mode: "form" }
#@markdown ## 1 · Project
#@markdown Select a profile for convenient starting values. All fields remain editable.
profile = "Tessera · Portable" #@param ["Tessera · Portable", "Generic · C++", "Custom"]
repo_url = "https://github.com/Jue-dev/Tessera.git" #@param {type:"string"}
branch_or_ref = "issue-6/unwind-lifecycle" #@param {type:"string"}
#@markdown ## 2 · Git access
#@markdown Public repositories need no token. Private repos: Colab Secrets → GH_TOKEN → enable notebook access.
secret_name = "GH_TOKEN" #@param {type:"string"}
git_username = "x-access-token" #@param {type:"string"}
clone_depth = 1 #@param {type:"integer"}
clone_timeout_seconds = 240 #@param {type:"integer"}
#@markdown ## 3 · Runtime
test_timeout_seconds = 900 #@param {type:"integer"}
install_clang_if_missing = True #@param {type:"boolean"}
install_libcxx_for_clang = True #@param {type:"boolean"}
continue_after_failure = True #@param {type:"boolean"}
keep_workspace = True #@param {type:"boolean"}
#@markdown ## 4 · Setup (optional)
#@markdown One shell command per field; runs before tests. Leave blank if not needed.
setup_command_1 = "" #@param {type:"string"}
setup_command_2 = "" #@param {type:"string"}
setup_command_3 = "" #@param {type:"string"}
#@markdown ## 5 · Test matrix
#@markdown Format: Test name :: command :: needs=tool. The needs section is optional.
test_1 = "GCC · strict mock :: bash verify/windows_mock/run_windows_mock_tests.sh" #@param {type:"string"}
test_2 = "Clang · strict mock :: EXTRA_CXXFLAGS='-stdlib=libc++' CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh :: needs=clang++" #@param {type:"string"}
test_3 = "GCC · ASan + UBSan :: EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh" #@param {type:"string"}
test_4 = "" #@param {type:"string"}
test_5 = "" #@param {type:"string"}
#@markdown ## 6 · Reuse / saved settings
#@markdown Optional: save or load settings JSON from Google Drive. Drive mount is requested only when enabled.
use_drive_settings = False #@param {type:"boolean"}
drive_settings_path = "/content/drive/MyDrive/ProjectTestConsole/settings.json" #@param {type:"string"}
load_saved_settings = False #@param {type:"boolean"}
save_settings_after_run = False #@param {type:"boolean"}

import os
import re
import json
import time
import shutil
import tempfile
import subprocess
import platform
import datetime
import pathlib
import urllib.parse
from google.colab import userdata

# The Colab form above is the configuration interface. This cell is the runner.
WORKDIR = pathlib.Path("/content/project-validation-workspace")
REPORT_PATH = pathlib.Path("/content/PROJECT_VALIDATION_REPORT.txt")
started_all = time.time()
issues, setup_results, test_results = [], [], []
commit = "NOT_AVAILABLE"
clone_ok = False
secret = None

def redact(value, token):
    value = str(value or "")
    if token:
        value = value.replace(token, "[REDACTED]")
    return value

def clip_output(value, limit=22000):
    value = str(value or "")
    if len(value) <= limit:
        return value
    half = (limit - 64) // 2
    return value[:half] + "\n\n[OUTPUT TRUNCATED]\n\n" + value[-half:]

def run_shell(command, cwd, env, timeout):
    start = time.time()
    try:
        result = subprocess.run(
            command,
            cwd=str(cwd),
            env=env,
            shell=True,
            executable="/bin/bash",
            capture_output=True,
            text=True,
            timeout=timeout
        )
        output = (result.stdout or "")
        if result.stderr:
            output += ("\n[STDERR]\n" if output else "") + result.stderr
        return result.returncode, round(time.time() - start, 2), output
    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout.decode("utf-8", "replace") if isinstance(exc.stdout, bytes) else (exc.stdout or "")
        stderr = exc.stderr.decode("utf-8", "replace") if isinstance(exc.stderr, bytes) else (exc.stderr or "")
        return 124, round(time.time() - start, 2), f"TIMEOUT after {timeout}s\n{stdout}\n[STDERR]\n{stderr}"
    except Exception as exc:
        return -1, round(time.time() - start, 2), f"{type(exc).__name__}: {exc}"

def parse_test(spec, index):
    spec = spec.strip()
    if not spec:
        return None, None
    parts = [part.strip() for part in spec.split("::", 2)]
    if len(parts) < 2 or not parts[0] or not parts[1]:
        return None, f"Test {index}: use 'Test name :: command :: needs=program'."
    needs = []
    if len(parts) == 3 and parts[2]:
        if not parts[2].startswith("needs="):
            return None, f"Test {index}: third section must begin with 'needs='."
        needs = [item.strip() for item in parts[2][6:].split(",") if item.strip()]
    return {"name": parts[0], "command": parts[1], "needs": needs}, None

def safe_repository_label(url):
    try:
        parsed = urllib.parse.urlsplit(url)
        if parsed.scheme and parsed.netloc:
            return urllib.parse.urlunsplit((parsed.scheme, parsed.netloc, parsed.path, "", ""))
    except Exception:
        pass
    return url.split("?", 1)[0]

def add_issue(stage, detail):
    issues.append({"stage": str(stage), "detail": clip_output(detail)})

# Optional configuration persistence on Google Drive.
CONFIG_FIELDS = ["profile", "repo_url", "branch_or_ref", "secret_name", "git_username", "clone_depth", "clone_timeout_seconds", "test_timeout_seconds", "install_clang_if_missing", "install_libcxx_for_clang", "continue_after_failure", "keep_workspace", "setup_command_1", "setup_command_2", "setup_command_3", "test_1", "test_2", "test_3", "test_4", "test_5"]
drive_config_path = pathlib.Path(drive_settings_path)
drive_ready = False
if use_drive_settings:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_config_path.parent.mkdir(parents=True, exist_ok=True)
        drive_ready = True
        if load_saved_settings and drive_config_path.exists():
            saved = json.loads(drive_config_path.read_text(encoding="utf-8"))
            for key in CONFIG_FIELDS:
                if key in saved:
                    globals()[key] = saved[key]
            print("Saved settings loaded:", drive_config_path)
    except Exception as exc:
        add_issue("Settings persistence", f"Drive settings unavailable: {type(exc).__name__}: {exc}")
# Validate basic configuration before launching any shell commands.
repo_url = str(repo_url).strip()
branch_or_ref = str(branch_or_ref).strip()
repo_label = safe_repository_label(repo_url)

if not repo_url or not (repo_url.startswith("https://") or repo_url.startswith("http://") or repo_url.startswith("git@")):
    add_issue("Configuration", "Repository URL must be https://, http://, or git@...")
if clone_depth < 0:
    add_issue("Configuration", "Clone depth must be 0 (full history) or a positive integer.")
if test_timeout_seconds < 1 or clone_timeout_seconds < 1:
    add_issue("Configuration", "Timeout values must be positive integers.")

test_specs = []
for position, raw_spec in enumerate([test_1, test_2, test_3, test_4, test_5], 1):
    parsed, error = parse_test(raw_spec, position)
    if error:
        add_issue("Configuration", error)
    elif parsed:
        test_specs.append(parsed)

if not test_specs:
    add_issue("Configuration", "No valid test commands were provided.")

if issues:
    print("Configuration issue(s) found. Correct the form fields and run this cell again.")
else:
    print("Project Test Console started.")
    print("Repository:", repo_label)
    print("Ref:", branch_or_ref or "(default branch)")
    print("Tests configured:", len(test_specs))

    # Read a Colab Secret only for Git clone. Public repositories can be cloned without it.
    try:
        if secret_name.strip():
            secret = userdata.get(secret_name.strip())
    except Exception:
        secret = None

    if shutil.which("git") is None:
        add_issue("Git", "git executable is not installed in this Colab runtime.")
    else:
        if WORKDIR.exists():
            shutil.rmtree(WORKDIR, ignore_errors=True)

        clone_env = os.environ.copy()
        clone_env["GIT_TERMINAL_PROMPT"] = "0"
        clone_env["GCM_INTERACTIVE"] = "Never"
        askpass_dir = None

        if secret:
            askpass_dir = pathlib.Path(tempfile.mkdtemp(prefix="project_test_askpass_"))
            askpass_file = askpass_dir / "askpass.sh"
            askpass_file.write_text(
                '#!/bin/sh\n'
                'case "$1" in\n'
                '  *Username*) printf "%s\\n" "$PROJECT_GIT_USERNAME" ;;\n'
                '  *) printf "%s\\n" "$PROJECT_GIT_TOKEN" ;;\n'
                'esac\n',
                encoding="utf-8"
            )
            askpass_file.chmod(0o700)
            clone_env.update({
                "GIT_ASKPASS": str(askpass_file),
                "PROJECT_GIT_USERNAME": git_username.strip() or "x-access-token",
                "PROJECT_GIT_TOKEN": secret
            })

        clone_command = ["git", "clone"]
        if clone_depth > 0:
            clone_command += ["--depth", str(clone_depth)]
        if branch_or_ref:
            clone_command += ["--single-branch", "--branch", branch_or_ref]
        clone_command += [repo_url, str(WORKDIR)]

        print("\n[1/3] Cloning repository…")
        try:
            cloned = subprocess.run(
                clone_command,
                env=clone_env,
                capture_output=True,
                text=True,
                timeout=clone_timeout_seconds
            )
            clone_output = redact((cloned.stdout or "") + (cloned.stderr or ""), secret)
            if cloned.returncode == 0:
                clone_ok = True
                commit = subprocess.check_output(
                    ["git", "rev-parse", "HEAD"],
                    cwd=str(WORKDIR),
                    text=True
                ).strip()
                print("Repository ready ✓ | commit:", commit[:12])
            else:
                add_issue("Git clone", f"exit={cloned.returncode}\n{clone_output or '(no output)'}")
        except subprocess.TimeoutExpired:
            add_issue("Git clone", f"Clone timed out after {clone_timeout_seconds} seconds.")
        except Exception as exc:
            add_issue("Git clone", f"{type(exc).__name__}: {exc}")
        finally:
            if askpass_dir:
                shutil.rmtree(askpass_dir, ignore_errors=True)
            clone_env.pop("PROJECT_GIT_TOKEN", None)
            clone_env.pop("PROJECT_GIT_USERNAME", None)
            secret = None

    if clone_ok:
        run_env = os.environ.copy()

        # User-defined setup commands run in order and do not inherit the GitHub token.
        for pos, command in enumerate([setup_command_1, setup_command_2, setup_command_3], 1):
            command = str(command).strip()
            if not command:
                continue
            print(f"\n[SETUP {pos}] {command}")
            code, duration, output = run_shell(command, WORKDIR, run_env, test_timeout_seconds)
            record = {
                "name": f"Setup {pos}",
                "command": command,
                "exit": code,
                "duration": duration,
                "output": clip_output(redact(output, secret))
            }
            setup_results.append(record)
            if code != 0:
                add_issue(record["name"], record["output"] or f"exit={code}")
                if not continue_after_failure:
                    break

        setup_blocked = any(item["exit"] != 0 for item in setup_results) and not continue_after_failure

        # Install libc++ if a configured Clang test requests it.
        clang_uses_libcxx = any("stdlib=libc++" in spec["command"] for spec in test_specs)
        if not setup_blocked and install_libcxx_for_clang and clang_uses_libcxx:
            if shutil.which("clang++") is None or not pathlib.Path("/usr/include/c++/v1/expected").exists():
                print("\n[TOOLS] Installing libc++ headers/runtime…")
                install_cmd = "apt-get update -qq && apt-get install -y libc++-18-dev libc++abi-18-dev"
                code, duration, output = run_shell(install_cmd, WORKDIR, run_env, max(test_timeout_seconds, 600))
                setup_results.append({"name": "Install libc++ for Clang", "command": install_cmd, "exit": code, "duration": duration, "output": clip_output(output)})
                if code != 0:
                    add_issue("Install libc++ for Clang", output or f"exit={code}")
        # Clang installation is optional and performed before the Clang test.
        clang_needed = any("clang++" in spec["needs"] for spec in test_specs)
        if not setup_blocked and install_clang_if_missing and clang_needed and shutil.which("clang++") is None:
            print("\n[TOOLS] Installing Clang…")
            code, duration, output = run_shell(
                "apt-get update -qq && apt-get install -y clang",
                WORKDIR,
                run_env,
                max(test_timeout_seconds, 600)
            )
            setup_results.append({
                "name": "Install Clang",
                "command": "apt-get update -qq && apt-get install -y clang",
                "exit": code,
                "duration": duration,
                "output": clip_output(output)
            })
            if code != 0:
                add_issue("Install Clang", output or f"exit={code}")

        print("\n[2/3] Running configured tests…")
        stop_tests = setup_blocked
        for spec in test_specs:
            if stop_tests:
                test_results.append({
                    "name": spec["name"], "status": "SKIP", "exit": None,
                    "duration": 0, "command": spec["command"],
                    "output": "Skipped because an earlier setup step failed and continue-after-failure is disabled."
                })
                continue

            missing = [name for name in spec["needs"] if shutil.which(name) is None]
            if missing:
                test_results.append({
                    "name": spec["name"], "status": "SKIP", "exit": None,
                    "duration": 0, "command": spec["command"],
                    "output": "Required executable(s) not found: " + ", ".join(missing)
                })
                print("SKIP:", spec["name"], "| missing:", ", ".join(missing))
                continue

            print("RUN :", spec["name"])
            code, duration, output = run_shell(
                spec["command"], WORKDIR, run_env, test_timeout_seconds
            )
            status = "PASS" if code == 0 else "FAIL"
            record = {
                "name": spec["name"],
                "status": status,
                "exit": code,
                "duration": duration,
                "command": spec["command"],
                "output": clip_output(redact(output, secret))
            }
            test_results.append(record)
            print(f"{status}: {spec['name']} ({duration}s)")
            if code != 0:
                add_issue(spec["name"], record["output"] or f"exit={code}")
                if not continue_after_failure:
                    stop_tests = True

if use_drive_settings and drive_ready and save_settings_after_run:
    try:
        settings = {key: globals().get(key) for key in CONFIG_FIELDS}
        drive_config_path.parent.mkdir(parents=True, exist_ok=True)
        drive_config_path.write_text(json.dumps(settings, indent=2, ensure_ascii=False), encoding="utf-8")
        print("Settings saved:", drive_config_path)
    except Exception as exc:
        add_issue("Settings persistence", f"Save failed: {type(exc).__name__}: {exc}")
# The report is plain text in the Colab cell output, ready to copy into ChatGPT.
elapsed = round(time.time() - started_all, 2)
passed = sum(item["status"] == "PASS" for item in test_results)
failed = sum(item["status"] == "FAIL" for item in test_results)
skipped = sum(item["status"] == "SKIP" for item in test_results)
setup_failed = any(item["exit"] != 0 for item in setup_results)

if issues or failed or setup_failed or not clone_ok:
    overall = "FAIL"
elif skipped:
    overall = "PARTIAL"
elif passed:
    overall = "PASS"
else:
    overall = "FAIL"

commit_url = ""
github_match = re.search(r"github\.com[:/]([^/]+/[^/.]+?)(?:\.git)?$", repo_url)
if github_match and commit != "NOT_AVAILABLE":
    commit_url = f"https://github.com/{github_match.group(1)}/commit/{commit}"

lines = [
    "PROJECT VALIDATION REPORT",
    f"Overall: {overall}",
    f"Repository: {repo_label}",
    f"Branch: {branch_or_ref or '(default branch)'}",
    f"Commit: {commit}",
    f"Commit URL: {commit_url or 'not available'}",
    f"Timestamp UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",
    f"Environment: {platform.platform()} | Python {platform.python_version()}",
    f"Duration: {elapsed}s",
    f"Results: {passed} PASS / {failed} FAIL / {skipped} SKIP",
    "",
    "SETUP RESULTS:"
]
if not setup_results:
    lines.append("(none)")
for item in setup_results:
    lines += [
        f"[{'PASS' if item['exit'] == 0 else 'FAIL'}] {item['name']} | exit={item['exit']} | {item['duration']}s",
        f"COMMAND: {item['command']}",
        item["output"],
        ""
    ]

lines.append("TEST RESULTS:")
if not test_results:
    lines.append("(none)")
for item in test_results:
    lines += [
        f"[{item['status']}] {item['name']} | exit={item['exit']} | {item['duration']}s",
        f"COMMAND: {item['command']}",
        item["output"],
        ""
    ]

lines.append("ISSUE POOL:")
if not issues:
    lines.append("(empty)")
for index, issue in enumerate(issues, 1):
    lines += [
        f"--- ISSUE {index}: {issue['stage']} ---",
        issue["detail"],
        "--- END ISSUE ---",
        ""
    ]

lines += ["", "END OF REPORT"]
report_text = "\n".join(lines)
REPORT_PATH.write_text(report_text, encoding="utf-8")

if clone_ok and not keep_workspace:
    shutil.rmtree(WORKDIR, ignore_errors=True)

# Render a copy-friendly HTML report; plain text remains the canonical report.
import html as html_escape_module
from IPython.display import display, HTML, FileLink

HTML_REPORT_PATH = pathlib.Path("/content/PROJECT_VALIDATION_REPORT.html")
esc = html_escape_module.escape
status_color = {"PASS": "#16803d", "FAIL": "#b42318", "SKIP": "#667085", "PARTIAL": "#b54708"}.get(overall, "#667085")
test_rows = "".join(
    f"<tr><td><b>{esc(item['status'])}</b></td><td>{esc(item['name'])}</td><td>{esc(str(item['exit']))}</td><td>{esc(str(item['duration']))}s</td></tr>"
    for item in test_results
) or "<tr><td colspan='4'>No tests ran</td></tr>"
failed_details = "".join(
    f"<details open><summary><b>{esc(item['name'])}</b> — exit {esc(str(item['exit']))}</summary><pre>{esc(item['output'] or '(no output)')}</pre></details>"
    for item in test_results if item["status"] == "FAIL"
) or "<p>No failed tests.</p>"
issue_details = "".join(
    f"<details><summary>{esc(issue['stage'])}</summary><pre>{esc(issue['detail'])}</pre></details>"
    for issue in issues
) or "<p>No issues recorded.</p>"
full_report_html = f"""<!doctype html><html><head><meta charset="utf-8"><title>Project Validation Report</title>
<style>body{{font:14px/1.5 system-ui,sans-serif;color:#202124;margin:16px}}.summary{{border-left:6px solid {status_color};padding:10px 14px;background:#f6f8fa;margin-bottom:14px}}table{{border-collapse:collapse;width:100%;margin:8px 0 16px}}th,td{{text-align:left;border-bottom:1px solid #ddd;padding:8px}}pre{{white-space:pre-wrap;overflow-wrap:anywhere;background:#f6f8fa;padding:10px;border-radius:6px}}textarea{{width:100%;min-height:320px;font:12px/1.4 monospace;box-sizing:border-box}}details{{margin:8px 0}}.hint{{color:#555}}</style></head><body>
<div class="summary"><h2 style="margin:0;color:{status_color}">Overall: {esc(overall)}</h2>
<b>{passed} PASS / {failed} FAIL / {skipped} SKIP</b><br>
Repository: {esc(repo_label)}<br>Ref: {esc(branch_or_ref or '(default branch)')}<br>Commit: {esc(commit)}<br>Duration: {elapsed}s</div>
<h3>Test summary</h3><table><thead><tr><th>Status</th><th>Test</th><th>Exit</th><th>Duration</th></tr></thead><tbody>{test_rows}</tbody></table>
<h3>Failures</h3>{failed_details}<h3>Issues</h3>{issue_details}
<h3>Copyable full report</h3><p class="hint">Click inside the text box, press Ctrl+A (or ⌘A), then Ctrl+C (or ⌘C).</p>
<textarea readonly spellcheck="false">{esc(report_text)}</textarea></body></html>"""
HTML_REPORT_PATH.write_text(full_report_html, encoding="utf-8")
display(HTML(full_report_html))
print("\nDownload report files:")
display(FileLink(str(REPORT_PATH), result_html_prefix="TXT report: "))
display(FileLink(str(HTML_REPORT_PATH), result_html_prefix="HTML report: "))